# 📊 Exploratory Data Analysis & Diagnostics for Federated IoT Intrusion Detection
## Master's Thesis: *Xây dựng và đánh giá prototype hệ thống phát hiện xâm nhập IoT sử dụng Federated Learning trong môi trường dữ liệu non-IID*
**Author:** Nguyễn Việt Kỳ Quân (2026) | **Institution:** HCMUT - VNU-HCM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/comprehensive_eda.ipynb)

---

### 🎯 Research Objectives & Thesis Scope
This notebook provides a focused, publication-grade Exploratory Data Analysis (EDA) of the **CICIoT2023** network traffic dataset. Rather than descriptive profiling, every diagnostic directly answers a fundamental research challenge for our Federated Learning prototype:

1. **Tabular Quality & Heavy Tails:** Why standard scaling fails on bursty traffic and why $\log(1+x)$ + `RobustScaler` is required.
2. **Severe Class Imbalance:** Why standard Cross-Entropy collapses on stealthy intrusions ($IR \approx 147\times$) and why `MultiClassFocalLoss` is necessary.
3. **Federated Statistical Heterogeneity:** How Dirichlet label skew ($\alpha \in \{0.5, 0.1\}$) induces **Client Drift** across IoT edge gateways and why **FedProx** ($\mu = 0.05$) is essential.
4. **Direct Architectural Synthesis:** Mapping statistical findings to concrete modules in our PyTorch/Flower implementation.

---
### 🛠️ Step 1: Environment Setup & Data Loading
We set up reproducibility seeds, import project modules from `src.eda` and `src.data`, and load the dataset (with stratified sampling for fast, memory-safe execution).

In [ ]:
import os, sys, gc, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Lock random seed for 100% reproducibility
SEED = 42
np.random.seed(SEED)

# Mount Google Drive if running in Colab
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    if not os.path.exists('/content/FedLearning'):
        import subprocess
        subprocess.run(["git", "clone", "https://github.com/NVKQ2022/FedLearning.git", "/content/FedLearning"], check=True)
    os.chdir('/content/FedLearning')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Import modular EDA and data tools
from src.eda import (
    diagnose_data_quality,
    analyze_features,
    plot_feature_skewness,
    analyze_partition,
    plot_class_distribution,
    summarize_partitions,
    plot_partition_heterogeneity,
    export_dataset_audit
)
from src.data.preprocessor import TARGET_COLUMN, compute_balanced_class_weights
from src.data.partitioner import partition_iid, partition_dirichlet

# Locate dataset
candidate_paths = [
    '/content/drive/MyDrive/NguyenVietKyQuanKLTN/datasets/merged_CICIOT2023_data.csv',
    '/content/drive/MyDrive/FedLearning/merged_CICIOT2023_data.csv',
    'datasets/CICIOT2023/merged_CICIOT2023_data.csv',
    'datasets/merged_CICIOT2023_data.csv',
]
dataset_path = next((p for p in candidate_paths if os.path.exists(p) and os.path.getsize(p) > 10*1024*1024), None)

# Sample size: 100,000 for rapid interactive exploration, or None for full 5.11M rows
SAMPLE_SIZE = 100_000

if dataset_path:
    print(f'📁 Loading dataset from: {dataset_path}')
    if SAMPLE_SIZE:
        labels_all = pd.read_csv(dataset_path, usecols=[TARGET_COLUMN])[TARGET_COLUMN]
        from sklearn.model_selection import StratifiedShuffleSplit
        sss = StratifiedShuffleSplit(n_splits=1, train_size=SAMPLE_SIZE, random_state=SEED)
        sample_idx, _ = next(sss.split(np.zeros(len(labels_all)), labels_all))
        idx_set = set(sample_idx)
        del labels_all; gc.collect()
        df = pd.read_csv(dataset_path, skiprows=lambda i: i > 0 and (i - 1) not in idx_set)
    else:
        df = pd.read_csv(dataset_path)
else:
    print('⚠️ Real dataset not found locally. Generating lightweight synthetic fallback...')
    classes = ['DDoS', 'Benign', 'DoS', 'Recon', 'Spoof', 'Mirai', 'Web-based', 'Brute-force']
    probs = [0.375, 0.215, 0.143, 0.135, 0.083, 0.041, 0.005, 0.003]
    N = 10000
    y_synth = np.random.choice(classes, size=N, p=[p/sum(probs) for p in probs])
    df = pd.DataFrame({
        'Header_Length': np.random.uniform(20, 60, N),
        'Rate': np.random.exponential(500, N),
        'Tot sum': np.random.pareto(2, N) * 1000,
        'IAT': np.random.exponential(0.05, N),
        'AVG': np.random.uniform(100, 800, N),
        'Std': np.random.exponential(150, N),
        'Tot size': np.random.pareto(1.8, N) * 500,
        'Variance': np.random.exponential(20000, N),
        TARGET_COLUMN: y_synth
    })

print(f'✅ Loaded: {df.shape[0]:,} samples x {df.shape[1]} features (RAM: {df.memory_usage(deep=True).sum()/(1024**2):.1f} MB)')

---
### 🔍 Step 2: Tabular Data Quality & Feature Skewness
Tabular network flows present two major risks for neural networks:
1. **Numerical Anomalies:** Division-by-zero infinities ($\pm\infty$) and clock-drift negative times (`IAT < 0`).
2. **Heavy-Tailed Distributions:** High Fisher-Pearson skewness ($|\gamma_1| > 2$), where extreme flow bursts cause exploding gradients.

**Thesis Justification:** Justifies `TabularDataPreprocessor` using non-negative clipping, median imputation, $\log(1+x)$ compression, and `RobustScaler` (median & IQR) instead of standard scaling.

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET_COLUMN]

# 1. Data Quality Audit
quality = diagnose_data_quality(df[feature_cols])
print('=' * 65)
print(f'Total Rows: {quality["total_records"]:,} | Feature Columns: {quality["total_features"]}')
print(f'Missing Values (NaNs):       {"YES ⚠️" if quality["has_missing_values"] else "NONE ✅"}')
print(f'Infinite Values (±inf):      {"YES ⚠️" if quality["has_infinite_values"] else "NONE ✅"}')
print(f'Negative Physical Metrics:   {len(quality["columns_with_negatives"])} features')
print(f'Zero-Variance Features:      {len(quality["zero_variance_columns"])} features')
print(f'Training Safety Clearance:   {"TRAINING-READY ✅" if quality["is_training_safe"] else "PREPROCESSING REQUIRED ⚠️"}')
print('=' * 65)

# 2. Skewness Profiling
feature_stats = analyze_features(df[feature_cols])
heavy_tailed = feature_stats[feature_stats['is_heavy_tailed']]
print(f'\nDetected {len(heavy_tailed)} heavy-tailed features (|skewness| > 2.0).')

# Render publication skewness plot
fig_skew = plot_feature_skewness(
    stats_df=feature_stats,
    top_k=10,
    title='Top 10 Most Skewed Flow Features (|γ₁| > 2 Threshold)',
    figsize=(8, 4),
    show=True
)

---
### 🎯 Step 3: Class Imbalance & Loss Function Strategy
In IoT intrusion detection, high-rate volumetric floods (**DDoS**, **DoS**) dwarf stealthy application-layer intrusions (**Web-based**, **Brute-force**).

$$\text{Imbalance Ratio: } IR_c = \frac{\max_j N_j}{N_c}$$

**Thesis Justification:** With $IR \approx 147\times$, standard Cross-Entropy yields **0% recall** on stealthy intrusions. We deploy **`MultiClassFocalLoss` ($\gamma=2.0$)** with inverse-frequency class weights to suppress gradients from well-classified majority DDoS samples, and evaluate with **Macro-F1** and **Minority Recall**.

In [ ]:
class_names = sorted(df[TARGET_COLUMN].unique().tolist())
partition_eda = analyze_partition(y=df[TARGET_COLUMN].values, class_names=class_names)

# Render horizontal class distribution plot
fig_dist = plot_class_distribution(
    eda=partition_eda,
    title=f'CICIoT2023 Multi-Class Threat Profile (N={partition_eda["total_samples"]:,})',
    figsize=(8, 4),
    show=True
)

# Compute balanced class weights: w_c = N / (C * N_c)
y_indices = np.array([class_names.index(c) for c in df[TARGET_COLUMN].values])
weights = compute_balanced_class_weights(y_indices, num_classes=len(class_names), strategy='balanced')

# Formatted breakdown table
max_cnt = max(partition_eda['class_counts'].values())
breakdown = pd.DataFrame([
    {
        'Class': c,
        'Count': f"{partition_eda['class_counts'][c]:,}",
        'Share (%)': f"{partition_eda['class_percentages'][c]:.2f}%",
        'IR': f"{max_cnt / max(1, partition_eda['class_counts'][c]):.1f}x",
        'Balanced Weight (w_c)': f"{weights[i].item():.3f}"
    }
    for i, c in enumerate(class_names)
]).sort_values(by='Count', ascending=False)

display(breakdown)

---
### 🌐 Step 4: Federated Non-IID Statistical Heterogeneity & Client Drift
The core research problem of this thesis is statistical heterogeneity across IoT gateways:
- **IID (Uniform):** Each client observes an identical attack mixture ($H_{\text{norm}} \approx 1.0$).
- **Non-IID Dirichlet ($\alpha = 0.5$):** Moderate label skew across clients.
- **Non-IID Dirichlet ($\alpha = 0.1$):** Severe label skew where edge nodes have **zero samples** of rare intrusions ($H_{\text{norm}} < 0.5$).

**Thesis Justification:** Under $\alpha = 0.1$, local client gradients point in opposing directions, causing **Client Drift** and aggregation breakdown in FedAvg. This directly justifies deploying **FedProx** ($\mu = 0.05$) with a proximal penalty to stabilize convergence.

In [ ]:
NUM_CLIENTS = 5

# 1. Partition under IID, Dirichlet alpha=0.5, and Dirichlet alpha=0.1
p_iid = partition_iid(y_indices, num_clients=NUM_CLIENTS, seed=SEED)
p_dir05 = partition_dirichlet(y_indices, num_clients=NUM_CLIENTS, alpha=0.5, seed=SEED)
p_dir01 = partition_dirichlet(y_indices, num_clients=NUM_CLIENTS, alpha=0.1, seed=SEED)

s_iid = summarize_partitions(p_iid, y_indices, class_names=class_names)
s_dir05 = summarize_partitions(p_dir05, y_indices, class_names=class_names)
s_dir01 = summarize_partitions(p_dir01, y_indices, class_names=class_names)

# 2. Normalized Shannon Entropy Comparison (1.0 = Uniform IID, -> 0.0 = Extreme Skew)
entropy_df = pd.DataFrame({
    'Client': [f'Client {i}' for i in range(NUM_CLIENTS)],
    'IID (Stratified)': s_iid['entropy'].round(3),
    'Non-IID (α=0.5)': s_dir05['entropy'].round(3),
    'Non-IID (α=0.1)': s_dir01['entropy'].round(3),
})
print('📊 Normalized Shannon Entropy per Edge Client (H_norm):')
display(entropy_df)

# 3. Stacked Class Share Comparison (IID vs alpha=0.5 vs alpha=0.1)
fig, axes = plt.subplots(3, 1, figsize=(10, 8), dpi=120)
cmap = plt.get_cmap('tab10')
scenarios = [
    ('Scenario E2: Stratified IID (Uniform)', s_iid),
    ('Scenario E4: Dirichlet α = 0.5 (Moderate Skew)', s_dir05),
    ('Scenario E5: Dirichlet α = 0.1 (Severe Skew & Missing Classes)', s_dir01)
]

for idx, (title, s_df) in enumerate(scenarios):
    ax = axes[idx]
    clients = [f'Client {int(c)}' for c in s_df['client_id']]
    accum = np.zeros(len(clients))
    for i, c in enumerate(class_names):
        col = f'{c}_pct'
        if col in s_df.columns:
            widths = s_df[col].values
            ax.barh(clients, widths, left=accum, label=c, color=cmap(i % 10), height=0.6)
            accum += widths
    ax.set_xlim(0, 100)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Class Share (%)', fontsize=8)
    ax.xaxis.grid(True, linestyle='--', alpha=0.5)

axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, fontsize=8)
plt.tight_layout()
plt.show()

---
### 🏛️ Step 5: Strategic Thesis Decision Matrix
Direct mapping connecting every empirical EDA diagnosis to our prototype architecture and experimental benchmarks:

In [ ]:
matrix = pd.DataFrame([
    {
        'Statistical Finding': 'Heavy-Tailed Skewness (|γ₁| > 2 on IAT, Rates)',
        'Threat to Model': 'Exploding gradients, loss = NaN, dying ReLUs',
        'Prototype Solution': 'log1p(x) + RobustScaler (median & IQR)',
        'Code Module': 'src.data.preprocessor.TabularDataPreprocessor'
    },
    {
        'Statistical Finding': 'Severe Class Imbalance (IR = 147x on Brute-force)',
        'Threat to Model': 'Majority class collapse, 0% minority recall',
        'Prototype Solution': 'MultiClassFocalLoss (γ=2.0) + Balanced Class Weights',
        'Code Module': 'src.losses.focal_loss.MultiClassFocalLoss'
    },
    {
        'Statistical Finding': 'Dirichlet Label Skew (α = 0.1, missing classes)',
        'Threat to Model': 'Client Drift: local updates diverge and cancel out',
        'Prototype Solution': 'FedProx proximal regularization (μ = 0.05)',
        'Code Module': 'src.federated.client.FlowerIoTClient'
    },
    {
        'Statistical Finding': 'Edge Resource & Memory Constraints',
        'Threat to Model': 'BatchNorm running stats drift; edge OOM crashes',
        'Prototype Solution': 'BatchNorm-free TabularIoTMLP (128-64, ~15k params)',
        'Code Module': 'src.models.mlp.TabularIoTMLP'
    },
    {
        'Statistical Finding': '5.11M Dataset Footprint (991 MB)',
        'Threat to Model': 'Ray simulation memory exhaustion on single node',
        'Prototype Solution': 'Isolated client npz partitions + Flower gRPC mode',
        'Code Module': 'src.federated.scenario & run_grpc.py'
    }
])

display(matrix)

---
### 💾 Step 6: Export EDA Artifacts
Persisting structured audit results to `reports/eda/dataset_eda.json` and high-resolution publication charts to `reports/figures/eda/`.

In [ ]:
os.makedirs('reports/eda', exist_ok=True)
os.makedirs('reports/figures/eda', exist_ok=True)

# 1. Export JSON audit
audit = {
    'dataset': 'CICIoT2023',
    'total_samples': int(len(df)),
    'quality': quality,
    'class_distribution': partition_eda,
    'top_skewed_features': heavy_tailed['feature'].head(10).tolist(),
    'federated_heterogeneity': {
        'num_clients': NUM_CLIENTS,
        'iid_entropy_mean': float(s_iid['entropy'].mean()),
        'dirichlet_05_entropy_mean': float(s_dir05['entropy'].mean()),
        'dirichlet_01_entropy_mean': float(s_dir01['entropy'].mean()),
    }
}
audit_path = export_dataset_audit('reports/eda', audit, filename='dataset_eda.json')
print(f'✅ Exported audit to: {audit_path}')

# 2. Save publication figures at 300 DPI
fig_dist.savefig('reports/figures/eda/class_distribution.png', bbox_inches='tight', dpi=300)
fig_skew.savefig('reports/figures/eda/feature_skewness.png', bbox_inches='tight', dpi=300)
print('✅ Saved publication figures (300 DPI):')
print('   - reports/figures/eda/class_distribution.png')
print('   - reports/figures/eda/feature_skewness.png')

print('\n🎉 EDA Complete: Dataset validated and certified for Scenarios E1 - E5!')